# Module 3 -- Assistant's Memory
## Memory = a List of Dicts

Last week you learned that a bot's **prompt** is just strings you assemble. This week we
de-magic the next piece: **"it remembers the conversation."**

There is nothing magical about memory. A chatbot's memory is just a **list of small
dictionaries** that grows by one every time someone says something. By the end of this
notebook you will have built that memory object by hand and connected it to the real
`ChatBot.history` in `rag-starter/ribot/chatbot.py`.

**Today's new primitives:** dicts → lists → list-of-dicts → Booleans → `if/elif/else` →
a *gentle* `while` + `.append()`.


## 1. One chat turn is a **dict**

A dictionary maps **keys** to **values**. The shape we care about all term is one chat turn:
`{"role": ..., "content": ...}`. Run the cell and read a value back out with `turn["role"]`.

In [ ]:
# A dict: keys on the left, values on the right.
turn = {"role": "user", "content": "hi"}

print(turn)
print("role    ->", turn["role"])     # read a value by its key
print("content ->", turn["content"])
print("type    ->", type(turn))

### **Your turn.** 

Make a dict for the *assistant's* reply, then read its
`content` back out.

In [ ]:
# Faded: fill in the value for content, then read it back.
reply_turn = {"role": "assistant", "content": "Hello! How can I help?"}

print(reply_turn["role"])
print(reply_turn["content"])

## 2. A conversation is a **list** of dicts

A list holds many items **in order**. Index the first with `[0]`, the last with `[-1]`.
We start the conversation with one `system` message (the bot's instructions).

In [ ]:
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "hi"},
]

print("number of messages:", len(messages))
print("first message:", messages[0])          # index by position
print("first role:   ", messages[0]["role"])  # list -> dict -> value
print("last content: ", messages[-1]["content"])

Notice the chain: `messages[0]` gives a **dict**, then `["role"]` gives the **value**
inside it. List → dict → value. That two-step lookup is the whole trick.

## 3. Growing the list with `.append()`

`.append()` adds **one item to the end** of a list. This method makes the conversation *grow*.

In [ ]:
messages = [{"role": "system", "content": "You are a helpful assistant."}]
print("start: ", len(messages), "message")

messages.append({"role": "user", "content": "What's the weather?"})
print("after user turn:     ", len(messages), "messages")

messages.append({"role": "assistant", "content": "I can't check live weather yet!"})
print("after assistant turn:", len(messages), "messages")

for m in messages:
    print(f"  [{m['role']}] {m['content']}")

**Goldfish bug.** Below we *forget* to append. Watch the list stay frozen.

In [ ]:
messages = [{"role": "system", "content": "You are a helpful assistant."}]

user_says = "My name is Sam"
reply = f"(echo) {user_says}"
# OOPS: we printed a reply but never appended anything to memory.
print("bot >", reply)

print("messages still has only:", len(messages), "item -> the bot remembers nothing")

## 4. Booleans + `if / elif / else`: turning text into **commands**

A comparison like `user == "/exit"` is just `True` or `False` — a **Boolean**. We use
`if/elif/else` to branch: is this a command (`/exit`, `/show`) or a normal message?

In [ ]:
print("/exit" == "/exit")   # True
print("/exit" == "hello")   # False
print(type(True))

# NOTE: the `def ...` wrapper (and the `: str` labels) are Week 4 material — today,
# read only the if/elif/else INSIDE it.
def classify(user: str) -> str:
    """Decide what kind of input this is, using if/elif/else."""
    if user == "/exit":
        return "quit the loop"
    elif user == "/show":
        return "print the memory"
    else:
        return "a normal message to remember"

for sample in ["/exit", "/show", "hello there"]:
    print(f"{sample!r:14} -> {classify(sample)}")

## 5. The memory loop (a gentle `while`)

Now we put it together. A real terminal bot uses `input()` inside a `while True:` loop. Live
`input()` won't run cleanly in a notebook, so here we **feed in a scripted list of turns** to
see the *exact same logic* run end-to-end. The shape is identical to the real loop in
`ribot/chatbot.py`.

Watch the three moves each normal turn: **append the user dict → make a reply → append the
assistant dict.** The list grows by two every turn.

In [ ]:
# NOTE: the `def` wrapper is Week 4 material; `break` exits the loop, `continue`
# skips to the next turn. Today's focus is the three moves inside the loop.
def run_chat(scripted_inputs):
    """Same logic as a live while-loop, but fed a list so it runs in a notebook."""
    messages = [{"role": "system", "content": "You are a helpful assistant."}]

    for user in scripted_inputs:          # stands in for `while True: user = input(...)`
        user = user.strip()
        if user == "/exit":               # Boolean test -> stop
            print("(loop stops)")
            break
        elif user == "/show":             # branch: print memory
            for m in messages:
                print(f"  [{m['role']}] {m['content']}")
            continue
        else:                              # a normal message: remember it
            messages.append({"role": "user", "content": user})        # 1. remember user
            reply = f"(echo) {user}"                                    # 2. make a reply
            messages.append({"role": "assistant", "content": reply})   # 3. remember reply
            print(f"you > {user}")
            print(f"bot > {reply}\n")

    return messages

final = run_chat(["My name is Sam", "What did I just say?", "/show", "/exit"])
print("\nFinal memory has", len(final), "messages.")

### Hand-trace it

Before trusting the code, predict the list yourself. After `"My name is Sam"` and
`"What did I just say?"`, `messages` should hold **5** dicts: 1 system + 2 user + 2 assistant.
The `/show` command prints memory but adds nothing; `/exit` stops the loop. Run the cell above
and confirm your count matches.

## 6. This is `ChatBot.history`

The `messages` list you just built is exactly the memory inside the course bot. Open
`rag-starter/ribot/chatbot.py` and you'll see:

```python
class ChatBot:
    def __init__(self, system_prompt=DEFAULT_SYSTEM):
        self.history = [{"role": "system", "content": system_prompt}]  # <- your list

    def add_turn(self, role, content):
        self.history.append({"role": role, "content": content})        # <- your .append
```

`self.history` is the list. `add_turn` is the `.append`. Later in the term, **retrieved
document context (Week 6–7)** and **agent tool-results (Week 8)** get appended to this same
list. You just built the raw material of the entire app's memory.

> The `class` / `__init__` / `self` syntax around it is **Week 7's star** — for now,
> pattern-match the list and the `.append`, and ignore the wrapper.

## 7. The payoff: plug a **real reply** into YOUR loop

The echo bot proved the plumbing works. Now the upgrade — and it is **one line**. Replace

```python
reply = f"(echo) {user}"
```

with

```python
reply = chat(messages)
```

`chat()` is the same function the course assistant uses (`ribot/llm.py`). The key detail: we pass it **the whole `messages` list**, not just the latest input — so the model can use what you told it earlier. This works offline: the `FakeLLM` really does read the history.

If you have not connected this notebook yet, run this once in a terminal from inside
`rag-starter/`, then select **`rag-starter/.venv`** as the notebook kernel:

```bash
uv venv && uv pip install -e '.[dev]'
```

In [ ]:
from ribot.llm import chat

def run_chat_v2(scripted_inputs):
    """The SAME loop as run_chat — only move 2 of the three moves changed."""
    messages = [{"role": "system", "content": "You are a helpful assistant."}]

    for user in scripted_inputs:
        user = user.strip()
        if user == "/exit":
            print("(loop stops)")
            break
        elif user == "/show":
            for m in messages:
                print(f"  [{m['role']}] {m['content']}")
            continue
        else:
            messages.append({"role": "user", "content": user})        # 1. remember user
            reply = chat(messages)                                     # 2. REAL reply — sees the whole list
            messages.append({"role": "assistant", "content": reply})   # 3. remember reply
            print(f"you > {user}")
            print(f"bot > {reply}\n")

    return messages

final = run_chat_v2(["My name is Sam", "What's my name?", "/exit"])

**Read the second answer.** The bot used the *first* turn to answer the *second* question — because your loop kept the list. 

Test it:

In [ ]:
# Goldfish contrast: the SAME question, but history NOT included.
print(chat([
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What's my name?"},   # no earlier turns in the list
]))

The only difference between "remembers" and "goldfish" is **whether the earlier dict was in the list you sent**. Memory = the list. That is the entire week.

> **Floor:** run both cells and say out loud which line makes the difference. **Stretch:** add a `/reset` command to `run_chat_v2` that sets `messages` back to just the system dict — then show that after `/reset` the bot no longer knows your name.

## Recap & what's next

- A chat turn is a **dict**: `{"role": ..., "content": ...}`.
- A conversation is a **list** of those dicts — read with `messages[0]["role"]`.
- **`.append()`** grows the list; forgetting it = goldfish memory.
- **Booleans + `if/elif/else`** turn typed text into commands (`/exit`, `/show`).
- A gentle **`while`** loop ties it together (we'll consolidate loops in Week 4).
- Swap the echo for **`chat(messages)`** and your loop is a real assistant that **answers from its memory** — the exact upgrade you ship this week.

> **Before the in-person memory design argument (optional, ~20 min).** Your list cannot grow
> forever — Week 2's §5.5 named the ceiling (the **context window**), and the in-person session
> makes you pick a policy for it: keep all, sliding window, keep-first-plus-window, or
> summarize-then-drop. 